# EMIT / Sentinel-2 — Spectral Regression (Color Matching)

Runs independently of `Pairs_Extract.ipynb`. Reads extracted tile pairs from Drive,
fits per-tile S2→EMIT spectral regression, produces synthetic 32-band imagery,
and saves per-tile R² results as CSV for downstream filtering.

**Inputs**: tile pairs on Drive (produced by `Pairs_Extract.ipynb`)  
**Outputs**: regression-synthetic GeoTIFFs, R² CSV, diagnostic plots

## 1. Setup

In [3]:
import os, getpass
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

In [4]:
import os, textwrap

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

fatal: destination path 'HyperSpectralSuperResolution' already exists and is not an empty directory.
/content/HyperSpectralSuperResolution


In [5]:
!pip install -q numpy scipy scikit-learn rasterio matplotlib tqdm joblib earthaccess

In [ ]:
import json, glob
from pathlib import Path
from datetime import datetime
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from documentation.config import PipelineConfig
from documentation.pairs_artifacts import load_pipeline_config
from data.EMIT.emit_utils import closest_bands

from spectral.s2_to_emit import (
    fit_tile, plot_r2_spectrum, plot_spectral_match,
)

from documentation.report_builder import (
    ReportBuilder, R2Aggregator,
    plot_r2_histogram, plot_r2_per_band,
)

print("All imports OK.")

## 2. Mount Drive

In [7]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Parameters

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  USER PARAMETERS — edit these before running
# ═══════════════════════════════════════════════════════════════════════

DRIVE_ROOT = Path("/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches")
RUN_TAG    = "2026-03-24"   # must match the extraction run

MODE       = "downsample"   # "upsample" or "downsample"
SHOW_PLOTS = False          # set True for interactive exploration

# ═══════════════════════════════════════════════════════════════════════

In [ ]:
drive_base = DRIVE_ROOT / RUN_TAG
config_dict = load_pipeline_config(drive_base)
config = PipelineConfig.from_dict({
    **config_dict,
    "aoi_lat": 0.0, "aoi_lon": 0.0,        # dummy — not used here
    "drive_root": str(drive_base),
})

spectral_params = config.spectral_params

print("Spectral params from pipeline_config.yaml:")
pprint(spectral_params)
print(f"Mode: {MODE}")

## 4. Discover tile pairs from Drive

Scans the Drive folder tree for `manifest.csv` files produced by `Pairs_Extract.ipynb`.
Each manifest lists the tile paths for one AOI/pair.

In [ ]:
# Discover all manifests: {drive_base}/{aoi_slug}/{pair_id}/manifest.csv
manifests = sorted(drive_base.glob("*/*/manifest.csv"))
print(f"Found {len(manifests)} pair manifest(s) under {drive_base}")
for m in manifests:
    print(f"  {m.parent.parent.name} / {m.parent.name}")

## 5. Run spectral regression for all pairs

For each pair: fit per-tile regression, produce synthetic GeoTIFFs,
and collect R² results.

In [ ]:
from joblib import Parallel, delayed

all_r2_rows = []

for manifest_path in tqdm(manifests, desc="Pairs"):
    pair_dir = manifest_path.parent
    pair_id  = pair_dir.name
    aoi_slug_name = pair_dir.parent.name

    mf = pd.read_csv(manifest_path)
    if "emit_b32_tif" not in mf.columns or "s2_tif" not in mf.columns:
        print(f"  {aoi_slug_name}/{pair_id}: manifest missing required columns — skipping")
        continue

    synth_dir   = pair_dir / "regression_synth"
    matcher_dir = pair_dir / "regression_matchers"
    plots_dir   = pair_dir / "plots"
    synth_dir.mkdir(parents=True, exist_ok=True)
    matcher_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Read wavelengths from metadata
    wl_path = list(pair_dir.glob("metadata/*.wavelengths.json")) or \
              list(pair_dir.glob("**/*.wavelengths.json"))
    if wl_path:
        with open(wl_path[0]) as f:
            wl_nm = np.array(json.load(f))
    else:
        import rasterio
        first_emit = mf["emit_b32_tif"].dropna().iloc[0]
        with rasterio.open(first_emit) as src:
            wl_nm = np.array([float(d.split(" ")[0]) for d in src.descriptions if d])
        if len(wl_nm) == 0:
            print(f"  {aoi_slug_name}/{pair_id}: cannot determine wavelengths — skipping")
            continue

    first_b32 = mf["emit_b32_tif"].dropna().iloc[0]
    import rasterio
    with rasterio.open(first_b32) as src:
        n_bands_b32 = src.count
    if len(wl_nm) > n_bands_b32:
        meta_jsons = sorted(pair_dir.glob("metadata/tiles/*.json"))
        if meta_jsons:
            with open(meta_jsons[0]) as f:
                tile_meta = json.load(f)
            b32_idx = tile_meta.get("emit_b32_indices_0based")
            if b32_idx:
                wl_b32 = wl_nm[b32_idx]
            else:
                idxs_0, _ = closest_bands(wl_nm, list(config.emit_target_wavelengths_nm))
                b32_idx = sorted(set(idxs_0))
                wl_b32 = wl_nm[b32_idx]
        else:
            idxs_0, _ = closest_bands(wl_nm, list(config.emit_target_wavelengths_nm))
            b32_idx = sorted(set(idxs_0))
            wl_b32 = wl_nm[b32_idx]
    else:
        wl_b32 = wl_nm

    r2_agg = R2Aggregator(wavelengths_nm=wl_b32)

    # ── Build work items ────────────────────────────────────────────────
    work = []
    for _, row in mf.iterrows():
        tile_idx = int(row["idx"])
        s2_tif   = row["s2_tif"]
        b32_tif  = row.get("emit_b32_tif")
        if pd.isna(b32_tif) or pd.isna(s2_tif):
            continue
        if not Path(b32_tif).exists() or not Path(s2_tif).exists():
            continue
        out_name = Path(s2_tif).name.replace("_s2.tif", "_regression_synth.tif")
        out_path = synth_dir / out_name
        # if out_path.exists():
        #     continue
        work.append((tile_idx, s2_tif, b32_tif, str(out_path)))

    if not work:
        print(f"  {aoi_slug_name}/{pair_id}: all tiles already done or no valid tiles")
        continue

    # ── Worker function ─────────────────────────────────────────────────
    # Capture loop-scoped variables as defaults to avoid closure leaks
    def _process_tile(tile_idx, s2_tif, b32_tif, out_path_str,
                      _matcher_dir=matcher_dir, _plots_dir=plots_dir,
                      _pair_id=pair_id, _wl_b32=wl_b32,
                      _spectral_params=spectral_params, _mode=MODE):
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        try:
            matcher, stats = fit_tile(
                s2_tile_path=s2_tif,
                emit_b32_tile_path=b32_tif,
                **_spectral_params,
                mode=_mode,
                verbose=False,
            )
        except ValueError as e:
            return {"tile_idx": tile_idx, "status": "skipped", "error": str(e)}

        matcher.save(_matcher_dir / f"tile_{tile_idx:03d}_matcher.joblib")
        matcher.apply_to_tile(s2_tif, out_path=out_path_str)

        plot_spectral_match(
            s2_tif, out_path_str, b32_tif,
            title_suffix=f"tile {tile_idx:03d}",
            r2_mean=stats["r2_mean"],
            save_path=str(_plots_dir / f"{_pair_id}_tile{tile_idx:03d}_synth.png"),
            show=False,
            wavelengths_nm=_wl_b32,
        )
        plt.close("all")

        return {
            "tile_idx":    tile_idx,
            "status":      "ok",
            "r2_mean":     stats["r2_mean"],
            "r2_per_band": stats["r2_per_band"],
        }

    # ── Parallel execution ──────────────────────────────────────────────
    results_tiles = Parallel(n_jobs=4, prefer="processes")(
        delayed(_process_tile)(idx, s2, b32, out)
        for idx, s2, b32, out in work
    )

    n_fitted = 0
    for res in results_tiles:
        if res["status"] == "skipped":
            print(f"    tile {res['tile_idx']:03d}: SKIPPED — {res['error']}")
            continue
        n_fitted += 1
        r2_agg.add(res["tile_idx"], np.array(res["r2_per_band"]), res["r2_mean"])
        all_r2_rows.append({
            "aoi_slug":    aoi_slug_name,
            "pair_id":     pair_id,
            "tile_idx":    res["tile_idx"],
            "r2_mean":     res["r2_mean"],
            **{f"r2_band_{i:02d}": v for i, v in enumerate(res["r2_per_band"])},
        })

    if n_fitted > 0:
        r2_summary = r2_agg.summary()
        print(f"  {aoi_slug_name}/{pair_id}: {n_fitted} tiles, "
              f"R² mean={r2_summary['global_mean']:.4f}, "
              f"median={r2_summary['global_median']:.4f}")
        plot_r2_histogram(
            r2_agg.r2_mean,
            plots_dir / "r2_histogram.png",
            title=f"R² distribution — {pair_id}",
        )
        plot_r2_per_band(
            r2_agg.r2_per_band, r2_agg.wavelengths_nm,
            plots_dir / "r2_per_band.png",
        )
        plt.close("all")
    else:
        print(f"  {aoi_slug_name}/{pair_id}: no tiles fitted")

print(f"\nDone. {len(all_r2_rows)} tiles processed across {len(manifests)} pairs.")

## 6. Save R² results

Writes a single CSV with per-tile R² (mean + per-band) across all AOIs/pairs.
This CSV is the input for downstream R²-based filtering in SR training.

In [17]:
r2_df = pd.DataFrame(all_r2_rows)

r2_csv_path = drive_base / "r2_all_tiles.csv"
r2_df.to_csv(r2_csv_path, index=False)

print(f"R² CSV saved: {r2_csv_path}  ({len(r2_df)} tiles)")
print(f"\nR² mean summary:")
print(f"  Mean:   {r2_df['r2_mean'].mean():.4f}")
print(f"  Median: {r2_df['r2_mean'].median():.4f}")
print(f"  Min:    {r2_df['r2_mean'].min():.4f}")
print(f"  Max:    {r2_df['r2_mean'].max():.4f}")
print(f"  Std:    {r2_df['r2_mean'].std():.4f}")

r2_df.head(10)

R² CSV saved: /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-24/r2_all_tiles.csv  (6690 tiles)

R² mean summary:
  Mean:   0.6773
  Median: 0.7602
  Min:    0.0030
  Max:    1.0000
  Std:    0.2460


,aoi_slug,pair_id,tile_idx,r2_mean,r2_band_00,r2_band_01,r2_band_02,r2_band_03,r2_band_04,r2_band_05,...,r2_band_22,r2_band_23,r2_band_24,r2_band_25,r2_band_26,r2_band_27,r2_band_28,r2_band_29,r2_band_30,r2_band_31
0,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,23,0.618337,0.382497,0.557199,0.570589,0.584489,0.598229,0.647745,...,0.638341,0.641459,0.661672,0.661243,0.652553,0.647967,0.652323,0.645199,0.637189,0.630810
1,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,30,0.613614,0.464690,0.566273,0.573986,0.603236,0.610257,0.631240,...,0.641674,0.643857,0.635005,0.642977,0.633379,0.627176,0.624843,0.607776,0.593680,0.588700
2,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,31,0.627435,0.658499,0.686780,0.687937,0.695009,0.699213,0.713926,...,0.589247,0.589487,0.582949,0.595387,0.589788,0.587296,0.584083,0.575878,0.560408,0.559940
3,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,37,0.681207,0.689194,0.708709,0.705270,0.708980,0.711677,0.717920,...,0.659301,0.658890,0.659509,0.664515,0.657676,0.653253,0.658453,0.658248,0.644017,0.644304
4,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,38,0.655524,0.530223,0.652825,0.658758,0.685848,0.693523,0.708682,...,0.637899,0.638710,0.628686,0.650445,0.651311,0.644592,0.643211,0.636046,0.626159,0.622339
5,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,39,0.766605,0.681262,0.779310,0.776798,0.777111,0.779984,0.789694,...,0.736347,0.738122,0.768544,0.775171,0.769143,0.766059,0.767382,0.771051,0.764715,0.762420
6,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,43,0.904338,0.782178,0.844260,0.846820,0.875554,0.881541,0.916387,...,0.938816,0.936395,0.935617,0.932224,0.932020,0.933060,0.930560,0.929742,0.929314,0.366369
7,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,44,0.638660,0.471437,0.579242,0.587224,0.603274,0.612382,0.645136,...,0.642089,0.647499,0.687241,0.683542,0.677778,0.675002,0.688811,0.696671,0.692450,0.683543
8,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,45,0.588438,0.614362,0.615768,0.608441,0.601565,0.601709,0.607526,...,0.570991,0.570558,0.576301,0.588339,0.588060,0.586836,0.596030,0.601306,0.590304,0.579939
9,aoi_lat-12.0_lon-77.0,20240501T153107_T18LTM_20240429,46,0.743983,0.320538,0.695154,0.714961,0.742003,0.754126,0.782213,...,0.712104,0.716019,0.769527,0.771557,0.760914,0.754901,0.764263,0.774802,0.759715,0.756690


## 7. Global R² diagnostics

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of mean R² across all tiles
axes[0].hist(r2_df["r2_mean"], bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(r2_df["r2_mean"].median(), color="red", ls="--", label=f"median={r2_df['r2_mean'].median():.3f}")
axes[0].set_xlabel("Mean R²")
axes[0].set_ylabel("Count")
axes[0].set_title("R² distribution (all tiles)")
axes[0].legend()

# Per-AOI box plot
aoi_order = r2_df.groupby("aoi_slug")["r2_mean"].median().sort_values(ascending=False).index
r2_df_sorted = r2_df.set_index("aoi_slug").loc[aoi_order].reset_index()
r2_df_sorted.boxplot(column="r2_mean", by="aoi_slug", ax=axes[1], rot=90)
axes[1].set_title("R² by AOI")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean R²")
plt.suptitle("")

plt.tight_layout()
fig.savefig(str(drive_base / "r2_global_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTiles with R² >= 0.75: {(r2_df['r2_mean'] >= 0.75).sum()} / {len(r2_df)}")
print(f"Tiles with R² >= 0.80: {(r2_df['r2_mean'] >= 0.80).sum()} / {len(r2_df)}")


Tiles with R² >= 0.75: 3487 / 6690
Tiles with R² >= 0.80: 2722 / 6690
